In [153]:
from Z import is_prime, divs, prime_power, factor, gcd, lcm, phi
from functools import total_ordering
from nimber_ops import *
from transfinite import Ordinal
import numpy as np
import pandas as pd

In [154]:
W = Ordinal

In [183]:
# utility functions: collect into util.py later
def base(n : int, b : int, length :  int = 0) -> list[int]:
    ''' returns the base b expansion of n as a  list
    n = sum_j^N a_j*b^j returns [a_0, a_1, ..., a_N]
    optional length parameter to include leading zeros if desired,
    otherwise minimum possible length where highest digit is non-zero'''
    assert (n >= 0 and b > 1), ('positional base expansion defined for' 
                                'non-negative integers and positive bases')
    if length == 0:
        if n == 0: return [0]
        coeffs = []
        while n > 0:
            coeffs.append(n % b)
            n //= b
        return coeffs
    else:
        coeffs = base(n, b)
        while len(coeffs) < length:
            coeffs.append(0)
        return coeffs

def base_eval(coeffs : list[int], b : int) -> int:
    total = 0
    for coeff in coeffs[::-1]:
        total = coeff + b * total
    return total
  
def ord_decomp(ordinal : Ordinal | int) -> list:
    ''' returns [inf_1, inf_2, ..., inf_n, finite term]'''
    if isinstance(ordinal, int):
        assert ordinal >= 0
        return [ordinal]
    high = Ordinal(ordinal.exponent, ordinal.coefficient)
    terms = [high]
    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        terms.append(Ordinal(remainder.exponent, remainder.coefficient, 0))
        remainder = remainder.addend
    assert(isinstance(remainder, int))
    terms.append(remainder)
    return terms

def ord_recomp(list : list) -> Ordinal | int:
    result = 0
    terms = sorted(list, reverse=True)
    for term in list:
        result = result + term
    return result

def ord_subtract(a, b):
    """
    Given ordinals a > b, return the ordinal c such that b + c == a

    """
    if a ==  b: return 0
    
    if a < b:
        raise ValueError("First argument must be greater than second argument")

    if type(a) == int:
        return a - b

    if type(b) == int or a.exponent > b.exponent:
        return a

    if a.exponent == b.exponent and a.coefficient == b.coefficient:
        return ord_subtract(a.addend, b.addend)

    # Here we know that a.coefficient > b.coefficient
    return Ordinal(a.exponent, a.coefficient - b.coefficient, a.addend)

def ord_div(ord1 : Ordinal, ord2 : Ordinal):
    ''' assuming both are single term: (w^a)*c / w^b = (w^(a - b))*c'''
    assert ord1 >= ord2
    if ord2 == 1: return ord1
    ex1, ex2 = ord1.exponent, ord2.exponent
    c = ord1.coefficient
    return Ordinal(ord_subtract(ex1, ex2), c)

def int_log(N: int, base: int) -> int:
    ''' Returns p such that base ^ p <= N < base ^ (p+1)'''
    def level(N: int, p: int = 2) -> int:
        '''
        Returns the largest 'level' L (w.r.t. p) such that 
        p ** (2 ** L) <= N
        '''
        L = 0
        if N < p:
            return 0
        while N // (p ** (1 << L)) != 0:
            L += 1
        return L-1

    if N < base:
        return 0
    # for the highest power 'exp' s.t. base^exp <= N, find
    # the largest power of 2 that is less than or equal to exp
    total = 1 << level(N, base)
    # now divide N to recursively find the binary expansion of exp
    N = N // (base ** total)
    while N != 0:
        total += 1 << level(N, base)
        N = N // (base ** (1 << level(N, base)))
    return total - 1

def pi(n : int) -> int:
    ''' prime counting function: returns # primes < n'''
    with open('small_primes.txt') as file:
        num_less = 0
        for line in file:
            p = int(line)
            if p < n:
                num_less += 1
            else:
                return num_less
        raise ValueError('n is too big to calculate pi(n) by brute force')
            
            
def f(p):
    '''Lenstra's f function: 
    for prime p, f(p) is min{ n | p divides 2^n-1}
    It is always the case that f(p) divides p-1'''
    divisors = divs(phi(p))
    for div in divisors[:-1]:
        if ((1<<div) -1) % p == 0:
            return div
    return divisors[-1]


small_primes = [2,3,5,7,11,13,17,19,23,29,31,37,41,43,47,53,59,61,67,71,73,79,\
    83,89,97,101,103,107,109,113,127,131,137,139,149,151,157,163,167,173,179,181,191,193,197,199,211,223,227,229,233,239,241,251,257,263,269,271,277,281,283,293,307,311,313,317,331,337,347,349,353,359,367,373,379,383,389,397,401,409,419,421,431,433,439,443,449,457,461,463,467,479,487,491,499,503,509,521,523,541,547,557,563,569,571,577]

df = pd.read_csv('alpha_p.csv') # got them all calculated as high as possible!
w = Ordinal()
alpha_p = {}
for i in df.index:
    alpha_p[int(df.loc[i, 'p'])] = eval(df.loc[i, 'alpha_p'])

####### this part was for the inductive construction of alpha_p ##############
###############################################################################
# kappa_p = { # generator of smallest extension of prime degree p
#            # kappa_{p_^n} = w^{w^{k-1}*p^{n-1}} where k = |{primes < p}|
#            2: 2, 3 : Ordinal()}
# for i, prime in enumerate(small_primes[2:]):
#     kappa_p[prime] = Ordinal(Ordinal(i+1))
# alpha_p = { # (kappa_p)^p, e.g. w^3=2, [w^w]^5 = 4, [w^w^2]^7 = w+1, etc.
#            # very hard to calculate for higher p, based on http://www.neverendingbooks.org/on2-extending-lenstras-list/
#             2:3, 3:2, 5:4, 7:Ordinal()+1, 11:Ordinal(Ordinal())+1, 13:Ordinal()+4,
#             17: 16, 19:Ordinal(3)+4, 23:Ordinal(Ordinal(3))+1, 
#             29:Ordinal(Ordinal(2))+4, 31:Ordinal(Ordinal())+1, 37: Ordinal(3)+4,
#             41: Ordinal(Ordinal())+1, 43:Ordinal(Ordinal(2))+1, 
#             47:Ordinal(Ordinal(7))+1, 
#             53: Ordinal(Ordinal(4))+1, 
#             59:Ordinal(Ordinal(8))+1, 
#             61:Ordinal(Ordinal())+Ordinal(), 
#             67:Ordinal(Ordinal(3))+Ordinal(),
#             71:Ordinal(Ordinal(2))+Ordinal(Ordinal()),
#             73:Ordinal(3)+1,
#             79:Ordinal(Ordinal(4))+1,
#             83:Ordinal(Ordinal(11))+1,
#             89:Ordinal(Ordinal(3))+1,
#             97:Ordinal()+256,
#             101:Ordinal(Ordinal()*5)+1,
#             103:Ordinal(Ordinal(5))+Ordinal()}
# excess = [3,0,0,1,1,0,0,4,1,0,1,0,1,1,1,1,1,0,0,0,1,1,1,1,0,
#  1,0,1,0,0,1,0,1,0,1,0,1,4,1,0,1,0,0,0,0,0,0,1,1,1,
#  1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,0,
#  1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,0,0,0,
#  0,1,1,1,1,0] # use https://oeis.org/A380496/

In [185]:
@total_ordering
class Nim:
    """nimbers"""

    def __init__(self, n: int | Ordinal) -> None:
        """ordinal considered an a field element in On_2
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        """

        def exp2(level: int) -> int:
            return 1 << level

        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True
            if self.val < 2:
                self.field = 2  # smallest
                self.base = 0
                self.high = 0
                self.low = self.val
                self.level = 0
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.level = level
                self.field = exp2(exp2(level))
                self.base = exp2(exp2(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            assert n < Ordinal(Ordinal(Ordinal())), 'ordinals < omega^omega^omega are implemented'
            self.val = n
            self.isfinite = False
            # these attributes can be found, but not as useful for infinite nums
            self.field = None
            self.base = None
            self.high = None
            self.low = None
            # assert isinstance(
            #     n, Ordinal), 'An infinite nimber must be an ordinal'
            # self.val = n
            # self.isfinite = False
            # # find the smallest field containing n
            # if (k := n.exponent) < Ordinal():  # n = omega^k, k fintie => cubic extension
            #     power = 0
            #     while 3 ** power <= k:
            #         power += 1
            #     self.field = Ordinal(3**power)
            #     exp = 3**(power-1)
            #     self.base = Ordinal(exp)
            #     high = Ordinal(n.exponent - exp, n.coefficient,
            #                 0) if exp < n.exponent else n.coefficient
            #     remainder = n.addend
            #     while isinstance(remainder, Ordinal) and remainder.exponent > exp:
            #         high += Ordinal(remainder.exponent - exp,
            #                         remainder.coefficient, 0)
            #         remainder = remainder.addend
            #         if isinstance(remainder, Ordinal) and remainder.exponent == exp:
            #             high += remainder.coefficient
            #             remainder = remainder.addend
            #     self.high = high
            #     self.low = remainder

    def __add__(self, other):
        if isinstance(other, int|Ordinal):
            return self + Nim(other)
        if self.isfinite and other.isfinite:
            return Nim(self.val ^ other.val)  # finite nim sum is bitwise XOR
        if self.val == other.val:
            return Nim(0)
        ord1, ord2 = self.val, other.val
        terms1, terms2 = ord_decomp(ord1), ord_decomp(ord2)
        sum = {}
        for term in terms1[:-1]:
            sum[term.exponent] = term.coefficient
        for term in terms2[:-1]:  # nim sum the coeffs if any terms with same exp
            try:
                sum[term.exponent] = sum[term.exponent] ^ term.coefficient
            except:
                sum[term.exponent] = term.coefficient

        keys = sorted(sum.keys())  # ordinal addition not commutative
        result = terms1[-1] ^ terms2[-1]  # nim sum of finite part
        for key in keys:
            if sum[key] > 0:  # add bigger on the left
                result = Ordinal(key, sum[key]) + result
        return Nim(result)

    def __eq__(self, other):
        if isinstance(other, int|Ordinal):
            return self == Nim(other)
        return self.val == other.val

    def __hash__(self):
        return self.val.__hash__()

    def __lt__(self, other):
        if isinstance(other, int|Ordinal):
            return self < Nim(other)
        return self.val < other.val

    def __mul__(self, other):
        assert isinstance(other, Nim|int|Ordinal)
        if isinstance(other, int|Ordinal):
            return self * Nim(other)
        x, y = self.val, other.val
        if x == 0 or y == 0:
            return Nim(0)
        if x == 1:
            return other
        if y == 1:
            return self
        if self.isfinite and other.isfinite:
            if self.val == other.val:
                return self.sq()

            def nim_product(a: int, b: int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low

                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a, q_b) * F_b ^ nim_product(a, r_b)
                    elif F_a > F_b:
                        return nim_product(q_a, b) * F_a ^ nim_product(r_a, b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a, q_b)
                        p_2 = nim_product(r_a, r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4

            return Nim(nim_product(x, y))
        elif self.isfinite and not other.isfinite:
            terms = ord_decomp(other.val)  # distribute to each coefficient
            terms[-1] = (self * Nim(terms[-1])).val
            for term in terms[:-1]:
                term.coefficient = (self * Nim(term.coefficient)).val
            return Nim(ord_recomp(terms))
        elif not self.isfinite and other.isfinite:
            return other * self
        else:  # both infinite
            terms1 = ord_decomp(self.val)
            if self.val == other.val and (len(terms1) > 2 or terms1[-1] > 0):
                return self.sq()
            terms2 = ord_decomp(other.val)
            inf1, fin1 = terms1[:-1], terms1[-1]
            inf2, fin2 = terms2[:-1], terms2[-1]
            if fin1 == 0 and fin2 == 0:
                to_sum = {Nim(0)}
                # result = Nim(0)

            else:
                # start by "FOIL-ing" to handle the terms where one is finite
                to_sum = (
                    {Nim(fin1) * other} ^ {self * Nim(fin2)} ^ {Nim(fin1) * Nim(fin2)}
                )
                # result = Nim(fin1) * other + self * Nim(fin2) \
                # + Nim(fin1) * Nim(fin2)
            for x in inf1:  # expand and distribute the purely infinite terms
                for y in inf2:
                    # calculate (w^N * a) x (w^M * b)
                    X, Y = sorted([x, y], reverse=True)  # X >= Y
                    N, M = X.exponent, Y.exponent  # N >= M
                    a, b = X.coefficient, Y.coefficient
                    coeff = Nim(a) * Nim(b)

                    if N < Ordinal() and M < Ordinal():  # handle finite case
                        N_tern = base(N, 3)  # write exponents in ternary
                        M_tern = base(M, 3)
                        K = max([len(N_tern), len(M_tern)])

                        # invert the powers of 3 since w^(3^k) = 2^(3^{-k-1})
                        def phi(n):
                            return base_eval(base(n, 3, K)[::-1], 3)  # reversed 3s

                        exp_p = phi(N) + phi(M)  # the exponent 2^(3^{-K} * exp_2)

                        next_power = 3**K
                        q, r = exp_p // next_power, exp_p % next_power
                        coeff = coeff * Nim(2) ** q
                        W = Nim(Ordinal(phi(r))) if r > 0 else Nim(1)
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    elif N < Ordinal() and not M < Ordinal():
                        W = Nim(Ordinal(N + M))
                        to_sum ^= {coeff * W}
                        # result = result + coeff * W
                    else:
                        assert max(N, M) < Ordinal(
                            105
                        ), "Nimber multiplication is only implemented for ordinals < w**w**105"
                        N_decomp, M_decomp = ord_decomp(N), ord_decomp(M)
                        # multiply all omega terms using exponent rules
                        N_fin_exp, M_fin_exp = N_decomp[-1], M_decomp[-1]
                        N_inf_exp, M_inf_exp = N_decomp[:-1], M_decomp[:-1]
                        # handle finite exponent first
                        omega_N = Ordinal(N_fin_exp) if N_fin_exp != 0 else 1
                        omega_M = Ordinal(M_fin_exp) if M_fin_exp != 0 else 1
                        # start multiplying omegas
                        prod: Ordinal = (Nim(omega_N) * Nim(omega_M)).val
                        # group the exponents by like terms
                        N_dict = {exp.exponent: exp for exp in N_inf_exp}
                        M_dict = {exp.exponent: exp for exp in M_inf_exp}
                        N_exp_exp = set(N_dict)
                        M_exp_exp = set(M_dict)
                        like_terms = N_exp_exp & M_exp_exp
                        unlike_terms = (N_exp_exp | M_exp_exp) - like_terms
                        for t in sorted(unlike_terms):  # normal ordinal product
                            mult = N_dict[t] if t in N_dict else M_dict[t]
                            prod = Ordinal(mult) * prod  # pull through
                        for n in like_terms:
                            i = N_dict[n].coefficient
                            j = M_dict[n].coefficient
                            p = small_primes[n + 1]
                            alpha = Nim(alpha_p[p])
                            i_p = base(i, p)  # write coeff in base p
                            j_p = base(j, p)
                            K = max([len(i_p), len(j_p)])

                            # invert the powers of p
                            def phi(n):
                                return base_eval(base(n, p, K)[::-1], p)

                            exp_p = phi(i) + phi(j)

                            next_power = p**K
                            q, r = exp_p // next_power, exp_p % next_power
                            coeff = coeff * alpha**q
                            term = Ordinal(Ordinal(n) * phi(r)) if r > 0 else 1
                            # need recursive because might be new like terms
                            prod = (Nim(prod) * Nim(term)).val
                        to_sum ^= {coeff * Nim(prod)}
                        # result = result + coeff * Nim(prod)
            return Nim(0).sum(*to_sum)

    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if p >= 0:  # binary exponentiation by squaring
            result = Nim(1)
            nimber = self
            while p > 0:
                if p & 1:
                    result = result * nimber
                nimber = nimber.sq()
                p >>= 1
            return result
        elif p == -1:
            if self.isfinite:
                if self.field == 2:
                    assert self.val > 0, 'inverse is not defined for zero'
                    return self
                a, b, F, f = (
                    Nim(self.high),
                    Nim(self.low),
                    Nim(self.base),
                    Nim(self.base >> 1),
                )
                det = (a + b) * b + a * a * f
                return det ** (-1) * (a * F + (a + b))
            # for infinite we need to invert via multiplication matrix 
            zero = Nim(0)
            one = Nim(1)

            # 2. Create the Augmented Matrix [A | e_0]
            A = np.array(self.matrix(), dtype=object)
            n = A.shape[0]
            
            b_col_list = [zero] * n
            b_col_list[0] = one
            b = np.array(b_col_list, dtype=object).reshape(n, 1) 
            
            Aug = np.concatenate((A, b), axis=1)
            
            # 3. Forward Elimination
            for k in range(n):
                # --- Pivoting ---
                pivot = Aug[k, k]
                
                if pivot == zero:
                    for i in range(k + 1, n):
                        if Aug[i, k] != zero:
                            Aug[[k, i]] = Aug[[i, k]]
                            break
                    else:
                        # Singular Matrix
                        return None 

                # --- Elimination ---
                pivot = Aug[k, k] # Re-get pivot in case of swap
                for i in range(k + 1, n):
                    if Aug[i, k] == zero:
                        continue
                    
                    factor = Aug[i, k] / pivot
                    
                    # **Single Vectorized Operation**
                    Aug[i, k:] = Aug[i, k:] + Aug[k, k:] * factor

            # 4. Back Substitution
            x = np.full(n, zero, dtype=object)

            for i in range(n - 1, -1, -1):
                # Sum of U[i, j] * x[j] for j > i
                if i < n - 1:
                    sum_ax = np.dot(Aug[i, i+1:n], x[i+1:])
                    val = Aug[i, n] + sum_ax 
                else:
                    val = Aug[i, n] # For the last row
                    
                x[i] = val / Aug[i, i]
            baseF = Nim(self.base_field())
            basis = np.array([baseF**k for k in range(n)], dtype=object)
            return np.dot(x, basis)
        else:
            inv = self ** (-1)
            return inv ** (-p)

    def __repr__(self) -> str:
        if self.isfinite:
            return str(self.val)
        else:
            return self.val.__repr__()

    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()

    def __truediv__(self, other):
        if not isinstance(other, int|Ordinal|Nim):
            raise ValueError('Quotient of nimbers must be be both nimbers')
        if isinstance(other, int|Ordinal):
            div = Nim(other)
        else:
            div = other
        assert div.val != 0, 'division by zero is not defined!'
        return self * div ** -1

    def as_vec(self, deg=None, return_basis=False) -> list|np.ndarray:
        ''' write the nimber as a vector n = a0*b0 + ... + aj*bj and return 
        [a0, ..., aj] where B = {b_j} is the basis {(nimber.base_field)^j}
        e.g.  7 = 3 + 1*4 returns [3, 1]
              omega^2 + omega*8 + 12 returns [12, 8, 1], etc. '''
        
        nimotize = np.vectorize(lambda x: Nim(x))
        if deg == None:
            if self.isfinite:
                vec = [Nim(self.low), Nim(self.high)]
                result = np.array(vec)
                basis  = np.array([Nim(1), Nim(self.base)])
                return result if not return_basis else [basis, result]

            # else infinite
            baseF, nextF = self.field_interval()
            dimension = (
                    nextF.exponent // baseF.exponent if baseF < Ordinal(Ordinal()) 
                    else nextF.exponent.coefficient // baseF.exponent.coefficient
                    )
            terms = ord_decomp(self.val)
            vec = np.full(dimension, 0, dtype=object)
            def ord_div(ord1 : Ordinal, ord2 : Ordinal):
                ''' assuming both are single term: (w^a)*c / w^b = (w^(a - b))*c'''
                assert ord1 >= ord2
                if ord2 == 1: return ord1
                ex1, ex2 = ord1.exponent, ord2.exponent
                c = ord1.coefficient
                return (Ordinal(ord_subtract(ex1, ex2), c) if ord_subtract(ex1, ex2)>0 
                        else c)
            j = 0
            for term in terms[::-1]:
                if term == 0: continue
                while term >= baseF ** (j + 1):
                    j+=1
                vec[j] = ord_div(term, baseF**j) + vec[j]
            
            if not return_basis: return nimotize(vec)
            else:
                basis = np.array([Nim(baseF)**k for k in range(dimension)])
                return [basis, nimotize(vec)]
            
        elif type(deg) == int:
            result = np.full(deg, 0, dtype='uint8')
            basis = std_basis(deg)
            for i, b in enumerate(basis[::-1]):
                if self >= b: 
                    self = self + b
                    result[i] = 1
            return result[::-1] if not return_basis else [basis, result[::-1]]
        else:
            raise ValueError(f'writing as a vector space over {deg} isn\'t implemeted')
        
    def base_field(self) -> Ordinal:
        ''' returns the greatest ordinal F <= self.val which is a field 
        under Nim operations. e.g. base_field(3)=2, base_field(omega)=omega, etc.'''
        return self.field_interval()[0]

    def deg(self, added_one=False) -> int:
        """returns the degree of the minimal polynomial"""
        if self.isfinite:
            return self.field.bit_length() - 1
        # if infinite, the kappa numbers are easier to calculate degree
        if self.is_field():
            return kappa_deg(self.is_field())
        # try writing as a sum of kappa terms
        terms = ord_decomp(self.val)
        if terms[-1] == 0: terms.pop() # remove trailing zero
        kappa_nums = [Nim(term).is_field() for term in terms]
        if all(kappa_nums):
            return lcm(*[kappa_deg(num) for num in kappa_nums])
        # last thing to try is adding one, since this doesn't change degree
        if added_one == False:
            return (self + Nim(1)).deg(added_one=True) # hopefully easier form
        
        # otherwise, we have no choice but to brute force. Might be very slow!
        d = 1
        x = self * self
        while x != self:
            x = x * x
            d += 1
        return d

    def det(self):
        """det(N) = determinant of the multiplication by N matrix
        This is the same as the field norm over the next smallest field.
        For x < omega, this is equivalent to
        det(x) = x^(x.base + 1) = x^(2^2^n + 1)  if 2^2^n <= x < 2^2^(n+1)
        
        For x>= omega, there might not be a next smallest field, so there
        is more choice for which subfield to express as a linear trans over
        """
        if self.isfinite:
            if self.field == 2:
                return self
            a, b, F = Nim(self.high), Nim(self.low), Nim(self.base >> 1)
            return (a + b) * b + a.sq() * F  # much faster than self**(self.base+1)
        # else
        A = self.matrix()
        n = A.shape[0]
        zero = Nim(0)
        one = Nim(1)
        
        # Forward Elimination
        det = one
        for k in range(n):
            # --- Pivoting ---
            pivot = A[k, k]
            
            if pivot == zero:
                # Search for swap
                for i in range(k + 1, n):
                    if A[i, k] != zero:
                        A[[k, i]] = A[[i, k]] # Vectorized swap
                        break
                else:
                    return zero # Singular Matrix

            pivot = A[k, k]
            det = det * pivot # Accumulate determinant

            # --- Elimination ---
            for i in range(k + 1, n):
                if A[i, k] == zero:
                    continue
                
                factor = A[i, k] / pivot
                
                # **Vectorized row operation**
                A[i, k:] = A[i, k:] + A[k, k:] * factor
        return det

    def det_star(self):
        """number of times x -> det(x) is repeated until landing in F_2"""
        iter = 0
        if self.level == 0:
            return iter
        d = self.det()
        while d != Nim(1):
            iter += 1
            d = d.det()
        return iter + 1

    def field_interval(self) -> list[Ordinal|int]:
        """returns (base, next), where base <= nimber < next and both are fields
        e.g. 2 -> (2, 4), 7 -> (4, 16), omega -> (omega, omega^3), etc. """
        if self.isfinite:
            return [self.base, self.field]
        
        # otherwise, only need to consider the highest power of Omega
        lead_exponent = ord_decomp(self.val)[0].exponent
        if lead_exponent < Ordinal():
            # field(w**n) = w^(3^K) where 3^(K-1)<= n < 3^K
            power = int_log(lead_exponent, 3)
            baseF = Ordinal(3**power)
            nextF = Ordinal(3**(power + 1))
            return [baseF, nextF]
        
        # if leading exponent is itself infinite, we need leading power of *that*
        lead_lead = ord_decomp(lead_exponent)[0]
        coef, ex = lead_lead.coefficient, lead_lead.exponent
        # which prime base we interpret coef depends on the exponent
        p = small_primes[ex + 1]
        # want the smallest power of p exceeding coef
        power = int_log(coef, p)
        # next smallest field is omega^(p^(next_power)omega^ex)
        baseF = Ordinal(Ordinal(ex, p**power))
        nextF = Ordinal(Ordinal(ex, p**(power+1)))
        return [baseF, nextF] 
        
    def inv(self):
        return self ** (-1)

    def is_gen(self) -> bool:
        """True iff order(self) == self.field - 1"""
        if self.isfinite:
            """is_gen ==> det* = self.level
            det* = self.level ==> is_gen if level < 6"""
            if self.val == 0:
                return False
            if self.level != self.det_star():
                # is_gen ==> det* = self.level
                return False
            elif self.level < 6:
                # self.base = 2^2^(n-1) + 1 is prime for n < 6
                return True
            elif self.level == 6:  # F_5 = 641 × 6,700,417
                level_small = (self**641).level
                level_big = (self**6700417).level
                return min(level_small, level_big) == 6  # both are divisors of order
            elif self.level == 7:  # F_7 = 274,177 × 67,280,421,310,721
                level_small = (self**274177).level
                level_big = (self**67280421310721).level
                return min(level_small, level_big) == 7 and self.det().is_gen()
            # could keep going, but the highest factored is 'only' F_11 anyways
            else:
                return self.order() == (1 << (1 << self.level)) - 1

    def is_field(self) -> int:
        ''' returns 0 if the ordinal [x] isn't a field under nim operations,
            returns p^n if [x] = kappa_{p^n}
        '''
        F = self.base_field()
        if F != self.val: 
            return 0
        if F < Ordinal():
            return 2 * int_log(F, 2) # kappa_{2^n} = [2^2^{n-1}]
        if F.exponent < Ordinal():
            p = 3
            n = int_log(F.exponent, p) # next field is kappa_{p^n}
        else:
            p = small_primes[(F.exponent).exponent + 1]
            n = int_log(F.exponent.coefficient, p)
        return p**(n+1)
    
    def matrix(self) -> np.ndarray:
        '''' returns the multiplication map of the nimber considered as a linear
        transform over the field [self.base] '''
        if self.isfinite:
            a, b = Nim(self.low), Nim(self.high)
            mat =  [[a,  b ], 
                    [b, a+b]]
            return np.array(mat, dtype=object)
        # else, use the basis {1, base, base^2, ..., base^(d-1)}
        baseF, nextF = self.field_interval()
        dim = (nextF.exponent // baseF.exponent if baseF < Ordinal(Ordinal()) 
               else nextF.exponent.coefficient // baseF.exponent.coefficient)
        kapp = Nim(baseF)
        alph = kapp**dim
        curr_row = np.array(self.as_vec(), dtype=object)
        mat = [curr_row]
        for _ in range(dim - 1):
            curr_row = np.roll(mat[-1], 1)
            curr_row[0] = curr_row[0] * alph
            mat.append(curr_row)
        return np.array(mat, dtype=object).T

    def next_field(self) -> Ordinal:
        return self.field_interval()[1]
        
    def order(self) -> int:
        if self.isfinite:
            if self.level <= 7:
                if self.is_gen():  # avoid infinite loop
                    return (1 << (1 << self.level)) - 1

            n = self.val
            L = self.level
            if L == 0:
                return n
            elif L == 1:
                return 3  # ord(2) = ord(3) = 3
            elif L <= 5:  # order divides 3 * 5 * 17 * 257 * 65_537
                return (self.base + 1) * self.det().order()
            else:
                factor = 1
                nimber = self
                while nimber.level > 5:
                    nimber = nimber.det()
                    factor *= nimber.base + 1

                prime_divs = nimber.order()
                nimber_to_primes = self**prime_divs  # order divides F_5*...*F_{L-1}
                factor *= prime_divs
                max_non_gen = (1 << (1 << L) - 1) // 3
                print(
                    f"Warning: Order of {self} may take forever to calculate:\
                        too difficult to factor {factor}"
                )
                for i in range(641, max_non_gen // prime_divs + 1, 2):
                    if (factor // prime_divs) % i:
                        continue
                    if (nimber_to_primes**i).val == 1:
                        return i * prime_divs
                return 1 << (1 << L) - 1
                # # make more efficient by only checking possible orders
                # # use Lagrange's theorem
                # # find the smallest field containing n i.e. smallest F_k > n
                # exp = (n.bit_length() - 1).bit_length()
                # # find the order of n must divide F_k - 1 which factors by difference of squares
                # divisors = fermat_divisors(exp)
                # for factor in divisors[:-1]:
                #     if (Nim(n)**factor).val == 1:
                #         return factor
                # else:
                #     return divisors[-1]
        else:
            ...  # inifite case is hard..

    def sum(self, *args):
        if len(args) == 0:
            return self
        elif len(args) == 1:
            return self + args[0]
        else:
            *inf, fin = ord_decomp(self.val)
            terms = {term.exponent: term.coefficient for term in inf}
            terms[0] = fin
            for arg in args:
                *inf_, fin_ = ord_decomp(arg.val)
                terms[0] ^= fin_
                for term_ in inf_:
                    try:
                        terms[term_.exponent] = (
                            terms[term_.exponent] ^ term_.coefficient
                        )
                    except:
                        terms[term_.exponent] = term_.coefficient
            result = terms[0]
            for key in sorted(terms.keys())[1:]:
                if terms[key] > 0:
                    result = Ordinal(key, terms[key]) + result
            return Nim(result)

    def sq(self):
        """returns the Nimber's square using Freshman's Dream"""
        if self.isfinite:
            a, b, base = self.high, self.low, self.base
            # x = a *2^2^n + b
            # x^2 = (a^2)*(2^2^n + 2^(2^n-1)) + b^2
            if self.base == 0:  # either 0 or 1
                return self
            else:
                term = base + (base >> 1)
                return Nim(a).sq() * Nim(term) + Nim(b).sq()
        else:
            terms = ord_decomp(self.val)
            sum = Nim(terms[-1]).sq()
            for term in terms[:-1]:
                sum = sum + Nim(term) * Nim(term)
            return sum

    def sqrt(self):
        if self.isfinite:
            if self.field == 2:
                return self
            term = self.sq() + self
            return term.sqrt() + self
        else:
            F = self.next_field()
            if F.exponent < Ordinal():
                p = 3
                n = int_log(F.exponent, p) # next field is kappa_{p^n}
            else:
                p = small_primes[(F.exponent).exponent + 1]
                n = int_log(F.exponent.coefficient, p)
            degree = kappa_deg(p**n)
            # x^2^d = x ==> sqrt(x) = x^2^(d-1)
            return self ** (1 << (degree - 1)) 

    def tr(self, over : int):
        deg = self.deg()
        assert deg % over == 0
        curr = self
        conjs = [self]
        for _ in range(1, deg):
            curr = curr ** (2**over)
            conjs.append(curr)
        return Nim(0).sum(*conjs)       
#-------------------------------------------------------------------------------#

def kappa(h : int) -> Nim:
    assert h > 0
    if h == 1 : result = Nim(0)
    elif prime_power(h):
        p, n = prime_power(h)
        k = pi(p)
        exp = Ordinal(k, h//p) if k > 0 else h//p
        result = Nim(2**exp)
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = kappa(g)
        else:
            result = kappa(q) + kappa(g)
    return result

def alpha(p):
    return Nim(alpha_p[p])
      
def Q(h : int) -> list[int]:
    ''' kappa(h) = sum_{q in Q} kappa(q), q prime powers'''
    assert h > 0
    if h == 1 : 
        result = []
    elif prime_power(h):
        result = [h]
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = Q(g)
        else:
            result = Q(g) + Q(q)
    return result

def kappa_deg(h : int) -> int:
    ''' faster way to calculate degree for kappa_h'''
    if h == 1: result = 1    
    elif prime_power(h):
        p, n = prime_power(h)
        if p == 2: result = h # deg(k_{2^n}) = 2^n
        else:
            result = h * lcm(kappa_deg(f(p)), (Nim(alpha_p[p]) + kappa(f(p))).deg())
    else:
        result = lcm(*[kappa_deg(q) for q in Q(h)])
    return result      
def smallest_of_same_order(nimber : Nim, order : int, small_by_terms=False, print_results=False) -> Nim:
    ''' given nimber and its order, find the least ordinal with the same order'''
    def is_less_by_terms(nim1: Nim, nim2 : Nim) -> bool:
        ''' order by number of terms in the cantor form rather than magnitude
            two ordinals with the same number of terms will be compared normally'''
        def num_terms(nim:Nim)->int:
            terms = ord_decomp(nim.val)
            num = len(terms) if terms[-1] > 0 else len(terms) - 1
            return num
        terms2 = num_terms(nim2)
        terms1 = num_terms(nim1)
        return terms1 < terms2 or (terms1==terms2 and nim1 < nim2)
    assert nimber ** order == 1
    deg = nimber.deg()
    curr_least = nimber
    candidate = nimber
    curr_least_pow = 1
    curr_pow = 1
    for n in range(2, order):
        if gcd(n, order)[-1] > 1: # won't have the same order
            continue 
        # try nimber ** n and check if it's less than current minimum
        # save compute using nimber ** n = (nimber**curr_pow)*(nimber**(n - curr_pow))
        # curr_pow isn't always n - 1 since we must skip when gcd > 1
        more_powers = n - curr_pow
        multiplicand = nimber ** more_powers
        candidate = candidate * multiplicand
        curr_pow = n
        
        condition = (is_less_by_terms(candidate, curr_least) if small_by_terms 
                        else (candidate < curr_least))
        if print_results:
            if n % 4096 == 0: print(f'tried up to {n}') # give periodic updates
        if condition:
            curr_least = candidate
            curr_least_pow = n
            if print_results:
                print(f'current least power is {curr_least_pow} : {curr_least}')
    return curr_least

def std_basis(n):
    assert kappa_deg(n) == n, (
        f'standard basis is only defined for dimensions n where deg(kappa_n) = n.'
        f'\nYou gave n={n}, which hass deg(kappa_{n})={kappa_deg(n)}')
    def even_basis(k):
        ''' basis for n = 2 ** k'''
        basis = np.arange(1<<k, dtype = object) # might get bigger than 64bit ints
        basis = 2 ** basis
        return np.vectorize(lambda x:Nim(x))(basis)
    def odd_basis(p, k):
        ''' basis for n = p ** k'''
        basis = np.full(p**k, kappa(p).val)
        powers = np.arange(p**k)
        basis = basis ** powers
        return np.vectorize(lambda x:Nim(x))(basis)
    
    factors = factor(n)
    result = np.array([Nim(1)])
    for p, k in factors[::-1]:
        subbasis = odd_basis(p, k) if p % 2 else even_basis(k)
        result = np.kron(result, subbasis)
    return result

def frob_matrix(deg:int) -> np.ndarray:
    ''' matrix form of x |-> x^2 over the field 2^deg'''
    deg = kappa_deg(deg)
    rows = [(nim**2).as_vec(deg) for nim in std_basis(deg)]
    return np.array(rows).T

def rewrite(vec, basis):
    """
    Rewrites a vector 'vec' in terms of the given 'basis' over the field F_2.
    
    This function solves the linear equation: coefficients @ basis = vec
    It uses Gauss-Jordan elimination on the augmented matrix [basis.T | vec.T].
    
    Args:
        vec: A (deg,) numpy array of 0's and 1's.
        basis: A (deg, deg) numpy array of 0's and 1's, where rows are basis vectors.
        
    Returns:
        A (deg,) numpy array of type 'uint8' containing the coefficients 
        of vec in the new basis.
    """
    deg = vec.shape[0]
    
    # We want to find coefficients 'c' such that: sum(c_i * basis_i) = vec
    # In matrix notation: c @ basis = vec
    # Taking the transpose: basis.T @ c.T = vec.T
    # This is a system A x = b, where A = basis.T, x = c.T, b = vec.T
    
    # Construct the augmented matrix M = [basis.T | vec.T]
    # We cast to uint8 strictly to ensure bitwise operations work as expected for GF(2)
    # basis.T ensures we are working with columns of the original basis (rows of the transpose)
    M = np.hstack((basis.T.astype(np.uint8), vec[:, np.newaxis].astype(np.uint8)))
    
    # Perform Gauss-Jordan elimination over F_2
    for i in range(deg):
        # --- 1. PIVOTING ---
        # We need a 1 at the diagonal position M[i, i]
        if M[i, i] == 0:
            # Look for a pivot in the column i, strictly below row i
            # np.argmax returns the index of the first '1' (max value) found
            # If the segment is all 0s, it returns 0 (handling this implicitly as no swap, 
            # though valid basis implies a pivot must exist)
            pivot_candidate_idx = np.argmax(M[i:, i])
            
            # The actual row index in M
            pivot_row = i + pivot_candidate_idx
            
            # Swap the current row with the pivot row
            # We use standard numpy indexing for swapping
            if M[pivot_row, i] == 1:
                M[[i, pivot_row], :] = M[[pivot_row, i], :]
        
        # --- 2. ELIMINATION ---
        # Eliminate 1s in the current column 'i' for all other rows
        # We want to XOR row i into any row j where M[j, i] == 1 (except row i itself)
        
        # Identify target rows: boolean mask where column i is 1
        rows_to_eliminate = M[:, i] == 1
        
        # Exclude the pivot row itself from elimination
        rows_to_eliminate[i] = False
        
        # Apply vectorized XOR row operation
        # This replaces the inner loop: row_j = row_j XOR row_i
        if np.any(rows_to_eliminate):
            M[rows_to_eliminate] ^= M[i]

    # After reduction, the left part is Identity (basis.T^-1 @ basis.T),
    # and the rightmost column is the solution x = c.T
    return M[:, -1]


In [157]:
def as_vec(nim:Nim, degree:int, return_basis=False)->np.ndarray|list:
    result = np.full(degree, 0, dtype='uint8')
    basis = std_basis(degree)
    for i, b in enumerate(basis[::-1]):
        if nim >= b: 
            nim = nim + b
            result[i] = 1
    return result[::-1] if not return_basis else [result[::-1], basis]
 
nimber = Nim(W(W(2)*6+2)+W(W(2))+1)
deg = 42

np.dot(*nimber.as_vec(deg=deg, return_basis=True))

w**(w**2*6 + 2) + w**w**2 + 1

In [158]:
nimber.deg()

42

In [159]:
a = np.array([1])
b = np.array([2, 3, 5])
np.kron(b, b)

array([ 4,  6, 10,  6,  9, 15, 10, 15, 25])

In [160]:
w = W()
nimber = Nim(eval('w**(w**6*18 + 7) + w**(w**6*18 + 6)*6 + w**(w**6*18 + 5)*13 + w**(w**6*18 + 4)*11 + w**(w**6*18 + 3)*12 + w**(w**6*18 + 2)*6 + w**(w**6*18 + 1)*4 + w**(w**6*18)*13 + w**(w**6*17 + 8)*7 + w**(w**6*17 + 7)*8 + w**(w**6*17 + 6)*14 + w**(w**6*17 + 5)*9 + w**(w**6*17 + 4)*8 + w**(w**6*17 + 3)*4 + w**(w**6*17 + 2)*15 + w**(w**6*17 + 1)*7 + w**(w**6*17) + w**(w**6*16 + 8)*12 + w**(w**6*16 + 7)*11 + w**(w**6*16 + 6)*13 + w**(w**6*16 + 5)*9 + w**(w**6*16 + 4)*5 + w**(w**6*16 + 3)*6 + w**(w**6*16 + 2) + w**(w**6*16 + 1)*5 + w**(w**6*16) + w**(w**6*15 + 8)*3 + w**(w**6*15 + 6)*14 + w**(w**6*15 + 5)*5 + w**(w**6*15 + 4)*8 + w**(w**6*15 + 3)*9 + w**(w**6*15 + 2)*5 + w**(w**6*15 + 1)*11 + w**(w**6*15)*8 + w**(w**6*14 + 8)*5 + w**(w**6*14 + 7)*4 + w**(w**6*14 + 6)*9 + w**(w**6*14 + 5)*3 + w**(w**6*14 + 4) + w**(w**6*14 + 2)*5 + w**(w**6*14 + 1)*4 + w**(w**6*14)*5 + w**(w**6*13 + 8)*4 + w**(w**6*13 + 7)*7 + w**(w**6*13 + 6)*13 + w**(w**6*13 + 5)*8 + w**(w**6*13 + 4)*8 + w**(w**6*13 + 3)*11 + w**(w**6*13 + 2)*12 + w**(w**6*13 + 1)*4 + w**(w**6*13)*7 + w**(w**6*12 + 8)*2 + w**(w**6*12 + 7) + w**(w**6*12 + 6)*11 + w**(w**6*12 + 5)*14 + w**(w**6*12 + 3)*9 + w**(w**6*12 + 2)*8 + w**(w**6*12 + 1) + w**(w**6*11 + 8)*13 + w**(w**6*11 + 7)*11 + w**(w**6*11 + 6)*12 + w**(w**6*11 + 5)*4 + w**(w**6*11 + 4)*11 + w**(w**6*11 + 3)*11 + w**(w**6*11 + 2)*12 + w**(w**6*11 + 1)*3 + w**(w**6*11)*7 + w**(w**6*10 + 8)*10 + w**(w**6*10 + 7)*9 + w**(w**6*10 + 6)*3 + w**(w**6*10 + 5)*13 + w**(w**6*10 + 4)*11 + w**(w**6*10 + 2)*9 + w**(w**6*10 + 1)*5 + w**(w**6*10)*4 + w**(w**6*9 + 8)*14 + w**(w**6*9 + 7)*13 + w**(w**6*9 + 6)*15 + w**(w**6*9 + 5)*10 + w**(w**6*9 + 4)*7 + w**(w**6*9 + 3)*10 + w**(w**6*9 + 2)*2 + w**(w**6*9 + 1)*2 + w**(w**6*9)*14 + w**(w**6*8 + 8)*6 + w**(w**6*8 + 6)*5 + w**(w**6*8 + 5)*15 + w**(w**6*8 + 4)*10 + w**(w**6*8 + 3)*11 + w**(w**6*8 + 2)*13 + w**(w**6*8 + 1)*2 + w**(w**6*8)*15 + w**(w**6*7 + 8)*7 + w**(w**6*7 + 7)*8 + w**(w**6*7 + 6)*12 + w**(w**6*7 + 5)*6 + w**(w**6*7 + 4) + w**(w**6*7 + 3)*6 + w**(w**6*7 + 2)*4 + w**(w**6*7 + 1)*13 + w**(w**6*7)*11 + w**(w**6*6 + 8)*8 + w**(w**6*6 + 7)*2 + w**(w**6*6 + 6)*4 + w**(w**6*6 + 5)*8 + w**(w**6*6 + 4)*4 + w**(w**6*6 + 3)*10 + w**(w**6*6 + 2)*10 + w**(w**6*6 + 1)*14 + w**(w**6*6)*12 + w**(w**6*5 + 8)*11 + w**(w**6*5 + 7)*10 + w**(w**6*5 + 6)*12 + w**(w**6*5 + 5)*14 + w**(w**6*5 + 4)*5 + w**(w**6*5 + 3)*3 + w**(w**6*5 + 2)*10 + w**(w**6*5 + 1)*14 + w**(w**6*5)*2 + w**(w**6*4 + 8)*9 + w**(w**6*4 + 7)*6 + w**(w**6*4 + 6)*8 + w**(w**6*4 + 5)*15 + w**(w**6*4 + 4)*3 + w**(w**6*4 + 3)*4 + w**(w**6*4 + 2)*5 + w**(w**6*4 + 1)*2 + w**(w**6*4)*10 + w**(w**6*3 + 8)*4 + w**(w**6*3 + 7)*4 + w**(w**6*3 + 6)*2 + w**(w**6*3 + 5)*9 + w**(w**6*3 + 4)*3 + w**(w**6*3 + 3)*5 + w**(w**6*3 + 2)*5 + w**(w**6*3 + 1) + w**(w**6*3)*15 + w**(w**6*2 + 8)*6 + w**(w**6*2 + 6)*2 + w**(w**6*2 + 5)*11 + w**(w**6*2 + 4)*14 + w**(w**6*2 + 3)*10 + w**(w**6*2 + 2)*10 + w**(w**6*2 + 1)*6 + w**(w**6*2)*13 + w**(w**6 + 8)*9 + w**(w**6 + 7)*7 + w**(w**6 + 6)*4 + w**(w**6 + 5)*5 + w**(w**6 + 4)*13 + w**(w**6 + 3)*5 + w**(w**6 + 2)*4 + w**(w**6 + 1)*8 + w**w**6*14 + 1'))

In [161]:

vec = Nim(W(2)*3+W()).as_vec(6)
basis_as_vec = np.array([Nim((W()+2))**(2**k) for k in range(6)])
basis_vecs = np.array([(Nim((W()+2))**(2**k)).as_vec(6) for k in range(6)])

np.dot(basis_as_vec, rewrite(vec, basis_vecs))

w**2*3 + w

In [162]:
def rand_nimber(degree : int) -> Nim:
    basis = kappa(degree).as_vec(deg=0, return_basis=True)[1]
    coeff = np.random.randint(0, 2, len(basis))
    return np.dot(basis, coeff)

def enum_nimbers(degree : int, start:Nim = Nim(0), end=None, reverse=False):
    basis = std_basis(degree)
    L = len(basis)
    start_vec = start.as_vec(degree)
    start_n = base_eval(start_vec, 2)
    end_n = 2**L if end is None else base_eval(end.as_vec(degree), 2)
    iter = range(start_n, end_n) if not reverse else reversed(range(start_n, end_n))
    for n in iter:
        coeff = base(n, 2, L)
        yield np.dot(basis, coeff)

def is_linearly_independent_f2(matrix: np.ndarray) -> bool:
    """
    Determines if the columns of an NxN binary matrix are linearly 
    independent over the field F2.

    Args:
        matrix (np.ndarray): An NxN numpy array containing only 0s and 1s.

    Returns:
        bool: True if columns are linearly independent (invertible), False otherwise.
    """
    # Operate on a copy to avoid modifying the original array
    # Ensure dtype is integer for bitwise operations
    m = matrix
    n_rows, n_cols = m.shape

    if n_rows < n_cols:
        return False

    # Perform Gaussian Elimination over F2
    for col in range(n_cols):
        # 1. Find a pivot in the current column at or below the diagonal
        # np.argmax returns the index of the first occurrence of the maximum value (1)
        pivot_row_offset = np.argmax(m[col:, col])
        pivot_row = col + pivot_row_offset

        # If the value at the pivot is 0, it means the whole column segment 
        # contains no 1s. The matrix is singular.
        if m[pivot_row, col] == 0:
            return False

        # 2. Swap the current row with the pivot row if necessary
        if pivot_row != col:
            m[[col, pivot_row]] = m[[pivot_row, col]]

        # 3. Eliminate 1s in rows below the pivot
        # Identify rows below the current one that have a 1 in this column
        rows_to_eliminate = m[col+1:, col] == 1
        
        # Apply XOR (addition in F2) to those rows
        # This utilizes NumPy broadcasting for speed
        m[col+1:][rows_to_eliminate] ^= m[col]

    # If we successfully found a pivot for every column, it is independent.
    return True

def is_normal(nimber:Nim, degree:int)->bool:
    vec = nimber.as_vec(degree)
    conjs = [vec]
    M = frob_matrix(degree)
    for _ in range(degree - 1):
        v = conjs[-1]
        next_conj = M@v % 2
        conjs.append(next_conj)
        if not is_linearly_independent_f2(np.array(conjs).T):
            return False
    return True

def last_nimber(deg:int)->Nim:
    deg = kappa_deg(deg)
    basis = std_basis(deg)
    return Nim(0).sum(*basis)

def conjugates(nimber, degree=None):
    deg = kappa_deg(nimber.deg()) if degree is None else degree
    matrix = [nim.as_vec(over=deg) for nim in [nimber**(2**n) for n in range(deg)]]
    return matrix



In [163]:
basis_as_vec = std_basis(20)
arr = np.random.randint(0, 2, 20)
Nim(0).sum(*basis_as_vec[arr==1])

w**(w*4)*4 + w**(w*3)*11 + w**(w*2)*15 + w**w*10 + 4

In [164]:
def fast_pow_2_to_the(nim:Nim, deg:int, powers:list[int], normal_basis_matrix=None, normal_basis_nim=None)->Nim:
    if type(powers) == int: powers = [powers]
    if normal_basis_matrix is None or normal_basis_nim is None:
        normal_basis_matrix, normal_basis_nim = normal_basis(deg)
    vec = nim.as_vec(deg)
    normal_coeff = rewrite(vec, normal_basis_matrix)
    to_the_powers = [np.roll(normal_coeff, power) for power in powers]
    def evaluate(arr)->Nim:
        return Nim(0).sum(*normal_basis_nim[arr==1])
    return [evaluate(coeff) for coeff in to_the_powers]

def normal_basis(deg):
    deg = kappa_deg(deg)
    normal = last_nimber(deg) # hopefully!
    F = frob_matrix(deg)
    std = std_basis(deg)
    basis_as_nim = [normal]
    basis_as_vec = [normal.as_vec(deg)]
    for _ in range(deg - 1):
        v = basis_as_vec[-1]
        next_vec = (F @ v) % 2
        basis_as_vec.append(next_vec)
        basis_as_nim.append(Nim(0).sum(*std[next_vec == 1]))
    normal_basis_matrix = np.array(basis_as_vec)
    normal_basis_nim = np.array(basis_as_nim)
    return normal_basis_matrix,normal_basis_nim

def fast_power(nim:Nim, deg:int, power:int, normal_basis_matrix=None, normal_basis_nim=None):
    bits = [i for i, bit in enumerate(base(power, 2)) if bit==1]
    result = Nim(1)
    powers = fast_pow_2_to_the(nim, deg, bits, normal_basis_matrix, normal_basis_nim)
    for nimber in powers:
        result = result * nimber
    return result


In [229]:
2**586//587

431458826419862083267384295611786580231643103031756259066440493081987663558924415642936958580430797392415699387121851249566946840085371787064339459666675645564915391278139549

In [165]:
nim11 = Nim(eval('w**(w**3*10 + w*2)*7 + w**(w**3*10 + w)*6 + w**(w**3*10)*6 + w**(w**3*9 + w*4)*2 + w**(w**3*9 + w*3)*15 + w**(w**3*9 + w*2)*4 + w**(w**3*9 + w)*14 + w**(w**3*9) + w**(w**3*8 + w*4)*10 + w**(w**3*8 + w*3)*11 + w**(w**3*8 + w*2)*6 + w**(w**3*8 + w)*2 + w**(w**3*8)*4 + w**(w**3*7 + w*4)*12 + w**(w**3*7 + w*3)*5 + w**(w**3*7 + w*2) + w**(w**3*7 + w)*15 + w**(w**3*7)*6 + w**(w**3*6 + w*4)*2 + w**(w**3*6 + w*3)*7 + w**(w**3*6 + w*2) + w**(w**3*6)*7 + w**(w**3*5 + w*4)*4 + w**(w**3*5 + w*3) + w**(w**3*5 + w*2)*9 + w**(w**3*5 + w)*6 + w**(w**3*5)*15 + w**(w**3*4 + w*4)*4 + w**(w**3*4 + w*3)*7 + w**(w**3*4 + w*2)*9 + w**(w**3*4 + w)*4 + w**(w**3*4)*2 + w**(w**3*3 + w*4)*9 + w**(w**3*3 + w*3)*9 + w**(w**3*3 + w*2)*14 + w**(w**3*3 + w)*3 + w**(w**3*3)*14 + w**(w**3*2 + w*4) + w**(w**3*2 + w*3)*14 + w**(w**3*2 + w*2) + w**(w**3*2 + w)*5 + w**(w**3*2)*7 + w**(w**3 + w*4)*5 + w**(w**3 + w*3)*4 + w**(w**3 + w*2) + w**(w**3 + w)*14 + w**w**3*3'))

In [205]:
def fast_tr(nimber, over : int, norm_basis_matrix, norm_basis_nim):
    deg = nimber.deg()
    assert deg % over == 0
    curr = nimber
    powers = [(over*k) for k in range(deg)]
    vec = nimber.as_vec(deg)
    normal_coeff = rewrite(vec, norm_basis_matrix)
    to_the_powers = [np.roll(normal_coeff, power) for power in powers]
    coeff = sum(to_the_powers) % 2
    return Nim(0).sum(*norm_basis_nim[coeff==1])

In [169]:
def last_is_normal(degree):
    for nim in enum_nimbers(degree, reverse=True):
        return is_normal(nim, degree)
# w = W()
# curr_least = Nim(eval('w**(w**2*4 + 2) + w**(w**2*4 + 1)*2 + w**(w**2*4)*3 + w**(w**2*3 + 1) + w**(w**2*3) + w**(w**2*2 + 2) + w**(w**2*2)*3 + w**(w**2 + 2)*3 + w**w**2*2 + w**2*2 + w*3 + 3'))
# while True:
#     nimber = rand_nimber(deg)
#     if nimber >= curr_least: continue
#     if is_normal(nimber, deg):
#         print(f'curr_least normal in 2^{deg}:  {nimber}')
#         curr_least = nimber

In [170]:
last_is_normal(kappa_deg(48))

True

In [171]:
def alpha_ord(p):
    alph = Nim(alpha_p[p])
    deg = alph.deg()
    N = ((1<<deg)-1)//p
    nimber = alph**p # p divides order of alpha_p
    for div in divs(N):
        if nimber**div == Nim(1):
            return p*div

# M = 50
# N = 60
# for p in small_primes[25:]:
#     if p == 151: continue
#     D = alpha(p).deg()
#     if M < D <= N:
#         try:
#             print(f'order of alpha_{p} is {alpha_ord(p)}')
#         except:
#             continue

In [172]:
# might save small amount of time with iterable instead of list
def ord_decomp_iter(ordinal):
    ''' Yields inf_1, inf_2, ..., inf_n, finite term one by one '''
    if isinstance(ordinal, int):
        yield ordinal
        return

    # Yield the highest term first
    yield Ordinal(ordinal.exponent, ordinal.coefficient)

    remainder = ordinal.addend
    while isinstance(remainder, Ordinal):
        yield Ordinal(remainder.exponent, remainder.coefficient, 0)
        remainder = remainder.addend

    assert isinstance(remainder, int)
    yield remainder

In [173]:
from Z import is_prime, divs, prime_power, factor, gcd, lcm


def lcm(*args):
    if len(args) == 0: return 0
    elif len(args) == 1: return args[0]
    else:
        n, m, *rest = args
    mult = n*m // gcd(n, m)[-1]
    for num in rest:
        mult = mult * num // gcd(mult, num)[-1]
    return mult

def pi(n : int) -> int:
    ''' prime counting function: returns # primes < n'''
    with open('small_primes.txt') as file:
        num_less = 0
        for line in file:
            p = int(line)
            if p < n:
                num_less += 1
            else:
                return num_less
        raise ValueError('n is too big to calculate pi(n) by brute force')
            
            
def f(p):
    '''Lenstra's f function: 
    for prime p, f(p) is min{ n | p divides 2^n-1}
    It is always the case that f(p) divides p-1'''
    assert(is_prime(p)), 'f(p) in only defined for primes p'
    if p in f_p:
        return f_p[p]
    divisors = divs(p-1)
    for div in divisors[:-1]:
        if ((1<<div) -1) % p == 0:
            f_p[p] = div
            return div
    f_p[p] = divisors[-1]
    return divisors[-1]

def kappa(h : int) -> Nim:
    assert h > 0
    if h in k_p:
        return k_p[h]
    if h == 1 : result = Nim(0)
    elif prime_power(h):
        p, n = prime_power(h)
        k = pi(p)
        exp = Ordinal(k, h//p) if k > 0 else h//p
        result = Nim(2**exp)
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = kappa(g)
        else:
            result = kappa(q) + kappa(g)
    k_p[h] = result
    return result
        
def Q(h : int) -> list[int]:
    ''' kappa(h) = sum_{q in Q} kappa(q), q prime powers'''
    assert h > 0
    if h in Q_p:
        return Q_p[h]
    if h == 1 : 
        result = []
    elif prime_power(h):
        result = [h]
    else:
        factors = factor(h)
        q = factors[0][0] ** factors[0][1]
        g = h // q
        if kappa_deg(g) % q == 0:
            result = Q(g)
        else:
            result = Q(g) + Q(q)
    Q_p[h] = result
    return result
def kappa_deg(h : int) -> int:
    ''' faster way to calculate degree for kappa_h'''
    if h in k_deg:
        return k_deg[h]
    if h == 1: result = 1
    
    elif prime_power(h):
        p, n = prime_power(h)
        if p == 2: result = h # deg(k_{2^n}) = 2^n
        else:
            result = h * lcm(kappa_deg(f(p)), (alpha(p) + kappa(f(p))).deg())
    else:
        result = lcm(*[kappa_deg(q) for q in Q(h)])
    k_deg[h] = result
    return result
    
def alpha(p : int) -> Nim:
    assert is_prime(p)
    if p in alpha_p:
        return Nim(alpha_p[p])
    else:
        if p in excess_p:
            alphaP = kappa(f(p)).val + excess_p[p]
            alpha_p[p] = alphaP
            return Nim(alphaP)
        d = kappa_deg(f(p))
        mersenne = ((1<<d) - 1)
        excess = 0
        # save some iterations with known lower bound for excess
        if len(Q(f(p))) == 1 and Q(f(p))[0] % 2 == 1:
            excess = 1 # Q(f(p)) = {q} odd prime power ==> excess >= 1
        if f(p) % 2 == 0 and prime_power(f(p)//2):
                if prime_power(f(p)//2)[0] == 3:
                    excess = 4 #f(p) = 2*3^k, k>0 ==> excess >=4           
        beta = Nim(kappa(f(p)).val + excess)
        expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        while expr:
            excess += 1
            beta = Nim(kappa(f(p)).val + excess)
            d = lcm(kappa_deg(f(p)), Nim(excess).deg())
            mersenne = ((1<<d) - 1)
            expr = (mersenne % p > 0) or beta**(mersenne//p) == Nim(1)
        alpha_p[p] = beta.val
        return beta
    

In [174]:
alpha(107)

w**w**14 + 1

In [175]:
k5 = Nim(W(W()))
g5 = k5+k5**(1<<5)+k5**(1<<10)+k5**(1<<15)
g5

w**(w*4)*10 + w**(w*3)*9 + w**(w*2)*8 + w**w

In [176]:
def tr_kappa(p):
    nimber = kappa(p)
    d = kappa_deg(p)
    result = nimber
    for q in range(1, d//p):
        nimber = nimber ** (1 << p)
        result = result + nimber
    return result

In [177]:
# d = {'p':excess_p.keys(), 
#      'kappa_p':[str(kappa(p)) for p in excess_p], 
#      'alpha_p': [str(alpha(p)) for p in excess_p], 
#      'excess' : [str(excess_p[p]) for p in excess_p],
#      'f(p)':[f(p) for p in excess_p],
#      'Q(f(p))': [Q(f(p)) for p in excess_p],
#      'identity_p': [f'$\\left[{kappa(p)._repr_latex_()[1:-1]}\\right]^{{{p}}}={alpha(p)._repr_latex_()[1:]}'for p in excess_p]}
# df = pd.DataFrame(d)
# df.to_csv('alpha_p(1).csv', index=True)

In [178]:
# let's compute higher alphas

# with open('small_primes.txt') as file:
#     for line in file:
        
#         p = int(line)
        
#         start = time.time()
#         alpha(p)
#         end = time.time()
#         alphaP = alpha_p[p]
#         df_p = pd.DataFrame({'p':p, 
#                             'kappa_p':[str(kappa(p))], 
#                             'alpha_P': [str(alpha_P[p])], 
#                             'excess' : [str(alpha(p)+kappa(f(p)))],
#                             'f(p)':[f(p)],
#                             'Q(f(p))': [Q(f(p))], 
#                             'compute_sec':end - start})
#         df = pd.concat([df, df_p], ignore_index=True)
#         df.to_csv('lenstra_nimber.csv', index=False)
#         print(f'alpha_{p}={alphaP}, was computed in {end - start}')
        
        # 47 takes too long :( my code for multiplication is too inefficient
        # also, the way factorizartion turns out yields **insane** exponent

In [179]:
excess = [3,0,0,1,1,0,0,4,1,0,1,0,1,1,1,1,1,0,0,0,1,1,1,1,0,
 1,0,1,0,0,1,0,1,0,1,0,1,4,1,0,1,0,0,0,0,0,0,1,1,1,
 1,0,0,1,0,1,1,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,1,0,0,
 1,1,1,0,0,0,0,1,0,1,0,0,1,1,0,1,1,1,0,1,1,0,0,0,0,
 0,1,1,1,1,0] # use https://oeis.org/A380496/